# S04: Dynamic OU pairs backtest (15-minute bars, stocks long/short)

Pre-registration: [`S04_README.md`](S04_README.md). Engine: `src/equity_trading/s04.py`.

1. Daily re-estimated pair list: rolling correlation pool, Engle–Granger on a re-estimated window, BH-FDR, 3-day hysteresis.
2. OU z-score trading: entry |z| ≥ 1.5, take profit |z| ≤ 0.5, stop |z| ≥ 3, time stop 3 × half-life, break exit.
3. Gates: block bootstrap, 1,000 random-pair and 1,000 random-timing placebos, double-cost run.

Set `ARM` below to `"hl5"` or `"fixed60"` and run once each.

In [ ]:
ARM = "fixed120"             # "hl5": clamp(5 x HL, 5, 60) sessions | "fixed60": 60 sessions | "fixed120": 120 sessions (A2)
N_PLACEBO = 1000             # gates are defined for 1000; lower it only for a first look
N_JOBS = -1                  # all CPU cores

import json, os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import __version__, s04
from equity_trading.alpaca import ALPACA_VERSION, download_years
from equity_trading.data import work_dir

SYNTHETIC = os.environ.get("S04_SYNTHETIC") == "1"
YEARS = list(range(2020, 2026))                      # 2026 sealed
ARMS = {"hl5": dict(window_rule="hl"),
        "fixed60": dict(window_rule="fixed", fixed_window_sessions=60),
        "fixed120": dict(window_rule="fixed", fixed_window_sessions=120, max_window_sessions=120)}  # A2: trading starts ~60 sessions later
CFG = s04.S04Config(**ARMS[ARM], n_placebo=N_PLACEBO)
if SYNTHETIC:
    YEARS, N_PLACEBO = [2024, 2025], 20

tag = ("synthetic_" if SYNTHETIC else "") + f"{ARM}_alpaca{ALPACA_VERSION}_s04{s04.S04_TESTS_VERSION}_p{s04.PANEL_VERSION}"  # test cache: test logic only
OUT = ROOT / "results" / ("s04_synthetic" if SYNTHETIC else "s04") / ARM
OUT.mkdir(parents=True, exist_ok=True)
CACHE = work_dir() / "results" / "s04" / tag
CACHE.mkdir(parents=True, exist_ok=True)
PANEL_CACHE = s04.panel_cache_path(CFG.timeframe)   # shared across studies, keyed on PANEL_VERSION

COL = {"strategy": "#2a78d6", "pairs": "#eb6834", "timing": "#1baf7a", "grey": "#8a8a84"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.titleweight": "bold", "font.size": 9})

if SYNTHETIC:
    BENCH = "XLK"
else:
    constituents = pd.read_csv(ROOT / "reference" / "sp500_constituents_20260913.csv")
    STOCKS = sorted(constituents["Symbol"].unique())
    BENCH = "SPY"
print(f"ARM={ARM} | synthetic={SYNTHETIC} | placebos={N_PLACEBO} | cache={CACHE}")
print(CFG)

## 1. Data: 15-minute OHLC grid and costs

Costs are a flat 2 bps one way (amendment 2026-11-03). The Abdi–Ranaldo estimate is plotted as a diagnostic only.

In [ ]:
t0 = time.time()
if SYNTHETIC:
    panel, STOCKS = s04.synthetic_ohlc_panel(YEARS)
else:
    download_years(STOCKS + [BENCH], CFG.timeframe, YEARS, workers=8)
    panel = s04.cached_panel(STOCKS + [BENCH], CFG.timeframe, YEARS, ffill_bars=CFG.ffill_bars)   # shared, versioned panel cache
TRADE_IDX = np.array([panel.symbols.index(s) for s in STOCKS if s in panel.symbols])
print(f"panel {panel.c.shape} ({panel.n_days} sessions) in {time.time() - t0:.0f}s | tradable symbols {len(TRADE_IDX)} | "
      f"missing from Alpaca: {sorted(set(STOCKS) - set(panel.symbols))}")

HS = panel.half_spread(CFG)
print(f"cost model: {CFG.cost_model}, one-way {np.median(HS) * 1e4:.2f} bps")
AR = panel.half_spread(s04.replace(CFG, cost_model="abdi_ranaldo"))
hs_bps = pd.DataFrame(AR[:, TRADE_IDX] * 1e4, index=pd.to_datetime(panel.days))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hs_bps.median(axis=1), color=COL["strategy"], lw=0.8, label="median")
ax[0].plot(hs_bps.quantile(0.9, axis=1), color=COL["grey"], lw=0.8, label="90th pct")
ax[0].set_title("Abdi–Ranaldo one-way estimate (bps, not used)"); ax[0].legend(frameon=False)
ax[1].hist(np.log10(hs_bps.median(axis=0).clip(lower=0.5)), bins=40, color=COL["strategy"])
ax[1].set_title("Per-stock median cost (log10 bps)")
plt.tight_layout(); plt.show()
print(hs_bps.stack().describe(percentiles=[.1, .5, .9]).round(2).to_dict())

## 2. Candidate pool and daily tests

The slow step. Results are cached per block of 20 decision days, so a re-run skips finished blocks.

In [ ]:
FIRST = s04.first_decision_day(CFG)
t0 = time.time()
pools = s04.candidate_schedule(panel, TRADE_IDX, CFG, FIRST)
pool_n = pd.Series({pd.Timestamp(panel.days[e]): len(p) for e, p in pools.items()})
print(f"pool built in {time.time() - t0:.0f}s: {len(pools)} decision days, pairs per day median {int(pool_n.median())}, "
      f"max {pool_n.max()}")
tests = s04.run_tests(panel, pools, CFG, cache_dir=CACHE, n_jobs=N_JOBS)
tests, spells = s04.tradable_spells(panel, tests, CFG, n_jobs=N_JOBS)
tests.to_parquet(CACHE / "tests_final.parquet")
spells.to_csv(OUT / "spells.csv", index=False)
print(f"test rows {len(tests):,} | raw p <= 1%: {(tests.p <= .01).mean():.4f} | BH pass: {tests.passed.mean():.5f} | "
      f"fail: {tests.failed.mean():.3f} | window < 5 HL: {tests.slow.mean():.3f}")

In [ ]:
D = panel.n_days
day_ix = pd.to_datetime(panel.days)
tradable_n = np.zeros(D, int)
for r in spells.itertuples(index=False):
    tradable_n[r.start:r.end] += 1
dur = spells.end - spells.start
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
ax[0].plot(day_ix, tradable_n, color=COL["strategy"], lw=0.8); ax[0].set_title("Tradable pairs per day")
by_day = tests.groupby("day").agg(raw=("p", lambda s: (s <= .01).mean()), bh=("passed", "mean"))
ax[1].plot(day_ix[by_day.index], by_day.raw, color=COL["grey"], lw=0.6, label="raw p <= 1%")
ax[1].plot(day_ix[by_day.index], by_day.bh, color=COL["strategy"], lw=0.8, label="BH pass")
ax[1].set_title("Daily pass rates"); ax[1].legend(frameon=False)
ax[2].hist(dur.clip(upper=120), bins=40, color=COL["strategy"]); ax[2].set_title("Spell length (sessions, capped 120)")
plt.tight_layout(); plt.show()
spell_stats = {"spells": int(len(spells)), "distinct_pairs": int(spells[["y", "x"]].drop_duplicates().shape[0]),
               "median_spell_sessions": float(dur.median()) if len(dur) else 0.0,
               "mean_tradable_per_day": float(tradable_n[FIRST:].mean()), "days_with_none": int((tradable_n[FIRST + 1:] == 0).sum())}
print(spell_stats)

## 3. Strategy backtest (1×, 2× and 0× cost)

In [ ]:
params = s04._Params(tests, len(panel.symbols))
t0 = time.time()
runs = {m: s04.backtest(panel, params, spells, panel.half_spread(s04.with_cost(CFG, m)), CFG, record=(m == 1.0))
        for m in (1.0, 2.0, 0.0)}
base = runs[1.0]
START = base["first_day"]
print(f"3 backtests in {time.time() - t0:.0f}s; trading starts {START.date()}")
metrics = pd.DataFrame({f"{m:g}x cost": s04.perf_metrics(r["daily_equity"], CFG.initial_equity, START) for m, r in runs.items()})
display(metrics.T.round(4))
tm = s04.trade_metrics(base["trades"]); display(pd.Series(tm).round(4).to_frame("strategy 1x"))
trades = base["trades"]
trades.drop(columns=["y", "x"], errors="ignore").to_csv(OUT / "trades.csv", index=False, float_format="%.6g")
base["daily_equity"].to_csv(OUT / "daily_equity.csv", header=["equity"], float_format="%.2f")

fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
for m, r in runs.items():
    e = r["daily_equity"][r["daily_equity"].index >= START]
    ax[0].plot(e.index, e, lw=0.9, label=f"{m:g}x cost", color={1.0: COL["strategy"], 2.0: COL["pairs"], 0.0: COL["grey"]}[m])
ax[0].set_title("Equity"); ax[0].legend(frameon=False)
e = base["daily_equity"][base["daily_equity"].index >= START]
ax[1].fill_between(e.index, e / e.cummax() - 1, 0, color=COL["pairs"], lw=0); ax[1].set_title("Drawdown (1x)")
plt.tight_layout(); plt.show()

In [ ]:
if len(trades):
    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    trades["reason"].value_counts().plot.bar(ax=ax[0], color=COL["strategy"]); ax[0].set_title("Exit reasons")
    ax[1].hist(trades["ret"].clip(-0.05, 0.05) * 100, bins=60, color=COL["strategy"]); ax[1].set_title("Trade net return (%, clipped)")
    no = pd.Series(base["n_open"], index=panel.index)
    no.groupby(no.index.date).max().plot(ax=ax[2], color=COL["strategy"], lw=0.8); ax[2].set_title("Open pairs (daily max)")
    plt.tight_layout(); plt.show()
    conc = pd.concat([trades["pair"].str.split("-").str[0], trades["pair"].str.split("-").str[1]]).value_counts()
    print("most traded symbols:", conc.head(10).to_dict())

## 4. Placebos: random pairs over the same spells, random contrarian timing

In [ ]:
from joblib import Parallel, delayed

pl_path = CACHE / f"placebos_{N_PLACEBO}_s04{s04.S04_VERSION}_{CFG.cost_model}{CFG.flat_half_spread_bps:g}bps.parquet"
if pl_path.exists():
    placebo = pd.read_parquet(pl_path)
else:
    t0 = time.time()
    jobs = [(m, CFG.seed * 1000 + i) for m in ("placebo_pairs", "placebo_timing") for i in range(N_PLACEBO)]
    placebo = pd.DataFrame(Parallel(n_jobs=N_JOBS, batch_size=4, verbose=5)(delayed(s04.run_placebo)(panel, params, spells, HS, CFG, m, base["entry_rate"], s, START) for m, s in jobs))
    placebo.to_parquet(pl_path); print(f"{len(jobs)} placebo backtests in {(time.time() - t0) / 60:.1f} min")
placebo.to_csv(OUT / "placebos.csv", index=False, float_format="%.6g")
print(f"strategy entry rate used by P-timing: {base['entry_rate']:.4f}")
bm = metrics["1x cost"]
ranks = {}
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for i, k in enumerate(["total_return", "sharpe"]):
    for mode, col in (("placebo_pairs", COL["pairs"]), ("placebo_timing", COL["timing"])):
        d = placebo.loc[placebo["mode"] == mode, k].to_numpy()
        ranks.setdefault(mode, {})[k] = s04.placebo_rank(bm[k], d)
        ax[i].hist(d, bins=40, alpha=0.6, color=col, label=mode.replace("placebo_", "P-"))
    ax[i].axvline(bm[k], color=COL["strategy"], lw=2, label="strategy"); ax[i].set_title(k); ax[i].legend(frameon=False)
plt.tight_layout(); plt.show()
display(pd.concat({m: pd.DataFrame(v).T for m, v in ranks.items()}).round(4))

## 5. Bootstrap, market exposure, calibration, gates

In [ ]:
r_daily = base["daily_equity"][base["daily_equity"].index >= START].pct_change().dropna().to_numpy()
boot = {b: s04.block_bootstrap_mean(r_daily, b, 10_000, CFG.seed) for b in (10, 20, 40)}
display(pd.DataFrame(boot, index=["mean", "lo95", "hi95"]).T)

bench_close = pd.Series(panel.c[panel.day_start[1:] - 1, panel.symbols.index(BENCH)], index=pd.to_datetime(panel.days))
mkt = s04.market_regression(base["daily_equity"][base["daily_equity"].index >= START], bench_close)
print("market regression vs", BENCH, {k: round(v, 5) for k, v in mkt.items()})

cal = s04.calibration(trades)
print("calibration (tp before stop):", {k: round(v, 4) for k, v in cal.items()})
if len(trades):
    res = trades[trades.reason.isin(["tp", "stop"])].copy()
    res["bucket"] = pd.cut(res.z_entry.abs(), [1.5, 1.75, 2.0, 2.5, 3.0], include_lowest=True)
    display(res.groupby("bucket", observed=True).agg(n=("reason", "size"), predicted=("p_tp_model", "mean"),
                                                     realized=("reason", lambda s: (s == "tp").mean())).round(3))

G = s04.gates(boot, ranks["placebo_pairs"], ranks["placebo_timing"], metrics["2x cost"].to_dict(), metrics["1x cost"].to_dict())
display(pd.Series(G).to_frame("S04 " + ARM))

In [ ]:
summary = {"study": "S04", "arm": ARM, "synthetic": SYNTHETIC, "years": YEARS, "config": CFG.as_dict(),
           "trading_start": str(START.date()), "n_placebo": N_PLACEBO, "spells": spell_stats,
           "metrics": metrics.to_dict(), "trades": tm, "calibration": cal, "market": mkt,
           "bootstrap": {str(k): v for k, v in boot.items()}, "placebo_ranks": ranks, "gates": G,
           "versions": {"package": __version__, "alpaca": ALPACA_VERSION, "s04": s04.S04_VERSION}}
# reported only: the other arms over this arm's trading period (later start for fixed120)
same_period = {}
for other in ARMS:
    f = OUT.parent / other / "daily_equity.csv"
    if other != ARM and f.exists():
        e = pd.read_csv(f, index_col=0, parse_dates=True)["equity"]
        same_period[other] = s04.perf_metrics(e / e[e.index < START].iloc[-1] * CFG.initial_equity if (e.index < START).any() else e,
                                              CFG.initial_equity, START)
if same_period:
    print(f"other arms from {START.date()} (rebased):"); display(pd.DataFrame(same_period).T.round(4))
summary["other_arms_same_period"] = same_period
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
(OUT / "manifest.json").write_text(json.dumps({"files": sorted(p.name for p in OUT.iterdir()), "cache": str(CACHE)}, indent=2))
print(json.dumps({k: G[k] for k in ["statistical_verdict", "economic_verdict", "prospective_verdict", "verdict"]}, indent=2))